# AKATSUKI Training — Kaggle (P100)

Settings → Accelerator → **GPU P100**
Settings → Internet → **On**

---

In [ ]:
import torch
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    gpu = torch.cuda.get_device_properties(0)
    print(f'GPU: {torch.cuda.get_device_name(0)} | VRAM: {gpu.total_mem/1e9:.1f}GB')
else:
    print('Enable GPU: Settings → Accelerator → GPU P100')

In [ ]:
# Kaggle has torch pre-installed with CUDA. Do NOT pip install torch.
!pip install -q transformers accelerate bitsandbytes peft trl datasets scipy pyyaml
print('Done')

In [ ]:
import os
if not os.path.exists('akatsuki'):
    !git clone https://github.com/kwkkd/akatsuki-hermes-integration akatsuki
%cd akatsuki

In [ ]:
from corpus_builder import build_corpus, build_dpo_corpus
build_corpus(num_articles=5000, num_dialogs=10000)
build_dpo_corpus(num_pairs=3000)

In [ ]:
!python continue_pretrain.py --model_id "deepseek-ai/DeepSeek-R1-Distill-Qwen-7B" --qlora --batch_size 2 --lr 5e-5 --epochs 1 --fp16

In [ ]:
!python train.py --model_id ./pretrained_akatsuki

In [ ]:
from inference import AkatsukiInference
from akatsuki_config import CONFIG
ai = AkatsukiInference(model_path=CONFIG.model.merged_path)
ai.load()

tests = ["너는 누구야?", "@Pain, C2 chain 구축해줘", "nmap SMB 취약점 스캔 명령어"]
for t in tests:
    print(f"=== {t} ===")
    print(ai.chat(t))
    print()

In [ ]:
!tar czf akatsuki_model.tar.gz merged_hacker_ai_model/
from IPython.display import FileLink
FileLink('akatsuki_model.tar.gz')